# LLM Agents (LangChain и LangGraph)

Небольшой overview

- в первой части реализован ReAct агент на базе LangChain, который помогает отслеживать дедлайны
- во второй части реализован Агент на базе LangGraph, который помогает пользователю отслеживать погоду (температура, влажность, ветер, прогноз на ближайшие дни)

In [ ]:
!pip install -qU langchain>=0.2.15 langchain-openai>=0.1.7 langgraph>=0.2.20 requests python-dotenv

In [ ]:
OPENROUTER_TOKEN = "" # заглушка

In [ ]:
# подключим модель
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    api_key=OPENROUTER_TOKEN,
    base_url="https://openrouter.ai/api/v1",
    model="gpt-4o-mini"
)

In [ ]:
# проверим, что все работает
response = llm.invoke("Привет! Как твои дела?").content
print(response)

Привет! У меня всё хорошо, спасибо! Как могу помочь тебе сегодня?


## Часть 1. ReAct агент на LangChain

### Краткое описание агента:

Данный агент помогает пользователю отслеживать свои учебные дедлайны. Агент позволяет пользователю хранить информацию о задачах и сроках их выполнения в одном месте, а также получать актуальную информацию о предстоящих и просроченных дедлайнах в диалоговом формате. Он может быть быть полезен в ситуациях, когда нужно хранить все дедлайны в одном месте, быстро получить список задач, определять ближайшие дедлайны, рассчитать количество дней до дедлайна и расставлять приоритеты.

### Какие tools использует агент
Агент использует следующие инструменты:
- добавление дедлайна
- удаление дедлайна
- вывод всех дедлайнов
- вывод дедлайнов по конкретному курсу
- вывод просроченных дедлайнов
- поиск ближайшего дедлайна
- расчёт количества дней до дедлайна
- изменение даты дедлайна

Инструменты реализованы в виде функций, обёрнутых с помощью декоратора @tool, что позволяет агенту вызывать их при необходимости.

### Использование памяти
Данный агент хранит историю запросов в `chat_history`, эта история передаётся в агент при каждом новом запросе. Модель использует историю напрямую и ещё получает её краткий анализ.

### Принцип работы агента
Агент реализован в соответствии с подходом ReAct (Reasoning + Acting).
При получении запроса пользователя он:
- Анализирует входной запрос и историю диалога
- Определяет, требуется ли использование одного из инструментов
- При необходимости вызывает соответствующий tool
- Получает результат выполнения инструмента
- Формирует итоговый ответ пользователю

Таким образом, агент не просто генерирует текст, а принимает решения о последовательности действий и использовании инструментов для решения задачи.

In [ ]:
# для начала сделаем хранилище дедлайнов, пусть это будет обычный список. Пример:
deadlines = [
    {
        "course": "Econometrics-2",
        "task": "Problem set 3",
        "deadline": "2026-03-23"
    },
    {
        "course": "Applied data analysis problems",
        "task": "Homework 1",
        "deadline": "2026-03-22"
    }
]

In [ ]:
deadlines = [] # пока очистим

Напишем функции агента и обернем их в tools

In [ ]:
from langchain.tools import tool

In [ ]:
@tool
def add_deadline(course: str, task: str, deadline: str):
  """Добавляет новый дедлайн. Используй формат даты YYYY-MM-DD"""
  deadlines.append({
        "course": course,
        "task": task,
        "deadline": deadline
    })
  return f"Дедлайн добавлен: {course} | {task} | {deadline}"

In [ ]:
@tool
def get_all_deadlines():
  """Возвращает все дедлайны"""
  if not deadlines:
        return "Список дедлайнов пуст."

  result = []

  for item in deadlines:
        result.append(f"{item['course']} | {item['task']} | {item['deadline']}")
  return "\n".join(result)

In [ ]:
@tool
def get_deadlines_by_course(course: str):
  """Возвращает дедлайны по указанному курсу"""
  filtered = list(filter(lambda x: x["course"].lower() == course.lower(), deadlines))

  if not filtered:
    return f"По курсу {course} дедлайнов не найдено"

  result = []
  for item in filtered:
    result.append(f"{item['course']} | {item['task']} | {item['deadline']}")
  return "\n".join(result)

In [ ]:
from datetime import datetime
@tool
def get_nearest_deadline():
  """Возвращает ближайший дедлайн"""
  if not deadlines:
    return "Список дедлайнов пуст"

  valid_deadlines = [] # список дата в формате datetime - задание
  for item in deadlines:
    dt = datetime.strptime(item["deadline"], "%Y-%m-%d")
    valid_deadlines.append((dt, item))

  nearest = min(valid_deadlines, key=lambda x: x[0])
  dt, item = nearest

  return f"Ближайший дедлайн: {item['course']} | {item['task']} | {item['deadline']}"

In [ ]:
@tool
def days_until_deadline(task: str):
  """Считает количество дней до дедлайна по названию задачи"""
  for item in deadlines:
    if item["task"].lower() == task.lower(): # ищем нужное задание
      today = datetime.today()
      deadline_date = datetime.strptime(item["deadline"], "%Y-%m-%d")
      delta = (deadline_date - today).days # считаем дни
      return f"До дедлайна '{item['task']}' осталось {delta} дней."

  return f"Задача '{task}' не найдена"

In [ ]:
@tool
def change_deadline(task: str, new_deadline: str):
  """Меняет дату дедлайна по названию задачи. Используй формат даты YYYY-MM-DD"""
  try:
    datetime.strptime(new_deadline, "%Y-%m-%d")
  except ValueError:
    return "Некорректный формат даты. Используй YYYY-MM-DD."

  for item in deadlines:
    if item["task"].lower() == task.lower():
      item["deadline"] = new_deadline
      return f"Дедлайн задачи '{task}' изменён на {new_deadline}."

  return f"Задача '{task}' не найдена"

In [ ]:
@tool
def missed_deadlines():
  """Выводит просроченные дедлайны"""
  today = datetime.today().date() # текущий момент времени
  missed = list(filter(lambda x: datetime.strptime(x["deadline"], "%Y-%m-%d").date() < today, deadlines))

  if not missed:
    return "Просроченных дедлайнов нет"

  return "\n".join(
        f"{item['course']} | {item['task']} | {item['deadline']}"
        for item in missed
    )

In [ ]:
@tool
def delete_deadline(course: str, task: str, deadline: str) -> str:
    """Удаляет дедлайн. Используй формат даты YYYY-MM-DD."""
    for item in deadlines:
        if (
            item["course"].lower() == course.lower()
            and item["task"].lower() == task.lower()
            and item["deadline"] == deadline
        ):
            deadlines.remove(item)
            return f"Дедлайн удалён: {course} | {task} | {deadline}"

    return "Такой дедлайн не найден"

In [ ]:
tools = [
    add_deadline,
    get_all_deadlines,
    get_deadlines_by_course,
    get_nearest_deadline,
    days_until_deadline,
    change_deadline,
    missed_deadlines,
    delete_deadline
]

Создаем агента

In [ ]:
from typing import Any
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import RunnableSerializable
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage

In [ ]:
# создадим промпт, который заставит LLM анализировать историю чата
history_analysis_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "Ты анализируешь историю диалога пользователя с агентом по дедлайнам. "
        "Выдели кратко важный контекст: какие курсы, задачи, дедлайны и последние намерения пользователя уже обсуждались. "
        "Если важного контекста нет, так и напиши."
    ),
    MessagesPlaceholder(variable_name="chat_history")
])

In [ ]:
# главный промпт для агента
agent_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "Ты — ReAct-агент, который помогает пользователю отслеживать учебные дедлайны. "
        "Ты умеешь использовать инструменты для добавления/удаления дедлайнов, просмотра списка задач, просмотра списков просроченных дедлайнов, "
        "поиска дедлайнов по курсу, изменения дат дедлайнов и поиска ближайшего дедлайна. "
        "Используй историю диалога и анализ истории, чтобы понимать контекст. "
        "Если для ответа нужен инструмент — вызови его. "
        "Если можно ответить без инструмента — ответь сам. "
        "Отвечай кратко, понятно и по делу."
    ),
    (
        "system",
        "Краткий анализ истории диалога:\n{history_analysis}"
    ),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])

In [ ]:
# конструктор агента
agent: RunnableSerializable = (
    {
        "input": lambda x: x["input"],
        "chat_history": lambda x: x["chat_history"],
        "agent_scratchpad": lambda x: x.get("agent_scratchpad", [])
    }
    | {
        "history_analysis": lambda x: llm.invoke(
            history_analysis_prompt.format_messages(
                chat_history=x["chat_history"]
            )
        ).content,
        "input": lambda x: x["input"],
        "chat_history": lambda x: x["chat_history"],
        "agent_scratchpad": lambda x: x.get("agent_scratchpad", [])
    }
    | agent_prompt
    | llm.bind_tools(tools, tool_choice="auto")
)

In [ ]:
name2tool = {tool.name: tool for tool in tools}

Создадим класс-обёртку, который управляет циклом работы агента.

In [ ]:
class CustomAgentExecutor:
    def __init__(self, agent, max_iterations: int = 5):
        self.max_iterations = max_iterations
        self.chat_history = []
        self.agent = agent

    def invoke(self, input: str) -> dict:
        count = 0
        agent_scratchpad = []

        while count < self.max_iterations:
            tool_call = self.agent.invoke({
                "input": input,
                "chat_history": self.chat_history,
                "agent_scratchpad": agent_scratchpad
            }) # при запросе история передается агенту

            agent_scratchpad.append(tool_call)

            if tool_call.tool_calls:
                final_answer_found = False

                for tool_call_obj in tool_call.tool_calls:
                    tool_name = tool_call_obj["name"]
                    tool_args = tool_call_obj["args"]
                    tool_call_id = tool_call_obj["id"]

                    if tool_name == "final_answer":
                        final_answer = tool_args["answer"]
                        final_answer_found = True
                        break

                    tool_out = name2tool[tool_name].invoke(tool_args)

                    tool_exec = ToolMessage(
                        content=f"{tool_out}",
                        tool_call_id=tool_call_id
                    )
                    agent_scratchpad.append(tool_exec)

                    #print(f"{count}: {tool_name}({tool_args}) -> {tool_out}")

                if final_answer_found:
                    break

                count += 1
            else:
                final_answer = tool_call.content
                break
        else:
            final_answer = "Достигнут лимит итераций без финального ответа."

        self.chat_history.extend([
            HumanMessage(content=input),
            AIMessage(content=final_answer)
        ]) # вот тут сохраняем историю после каждого запроса

        return {"output": final_answer}

In [ ]:
react_agent = CustomAgentExecutor(agent=agent, max_iterations=5)

Проверим то, как он работает

In [ ]:
print(react_agent.invoke("Добавь дедлайн: Econometrics-2 | Problem set 3 | 2026-03-23"))
print(react_agent.invoke("Добавь дедлайн: Applied data analysis problems | Homework 1 | 2026-03-22"))

{'output': 'Дедлайн добавлен: Econometrics-2 | Problem set 3 | 2026-03-23.'}
{'output': 'Дедлайн добавлен: Applied data analysis problems | Homework 1 | 2026-03-22.'}


In [ ]:
print(react_agent.invoke("Покажи все мои дедлайны"))

{'output': 'Вот все ваши дедлайны:\n\n1. **Econometrics-2** | Problem set 3 | 2026-03-23\n2. **Applied data analysis problems** | Homework 1 | 2026-03-22'}


In [ ]:
print(react_agent.invoke("Какой у меня ближайший дедлайн?"))

{'output': 'Ваш ближайший дедлайн: **Applied data analysis problems** | Homework 1 | 2026-03-22.'}


In [ ]:
print(react_agent.invoke("А по какому курсу он?"))

{'output': 'Ближайший дедлайн относится к курсу **Applied data analysis problems**.'}


In [ ]:
print(react_agent.invoke("Покажи мне просроченные дедлайны"))

{'output': 'Просроченных дедлайнов нет.'}


In [ ]:
print(react_agent.invoke("Удали дедлайн: Applied data analysis problems | Homework 1 | 2026-03-22"))
print(react_agent.invoke("Добавь дедлайн: Econometrics-2 | Test 5 | 2026-03-24"))
print(react_agent.invoke("Покажи все мои дедлайны"))

{'output': 'Дедлайн удалён: **Applied data analysis problems** | Homework 1 | 2026-03-22.'}
{'output': 'Дедлайн добавлен: Econometrics-2 | Test 5 | 2026-03-24.'}
{'output': 'Вот все ваши дедлайны:\n\n1. **Econometrics-2** | Problem set 3 | 2026-03-23\n2. **Econometrics-2** | Test 5 | 2026-03-24'}


Кажется все работает как надо

## Часть 2. Агент на LangGraph

### Краткое описание агента:

В этой части реализован агент на LangGraph, который помогает пользователю получать информацию о погоде в нужном городе. Агент умеет отвечать на вопросы о текущей погоде, температуре, влажности, ветре, а также давать прогноз на день и интерпретировать погодные условия в удобной для пользователя форме.

В основе решения лежит связка из:
- LangGraph для построения агентного workflow,
- открытого weather API,
- LLM для анализа и формулировки итогового ответа

### Принцип работы агента

Агент построен на LangGraph как последовательный workflow из нескольких этапов. Сначала он анализирует пользовательский запрос и определяет:
1. город,
2. тип погодного запроса,
3. какие погодные параметры нужно получить

После этого агент получает координаты города через geocoding API, затем отправляет запрос к открытому API прогноза погоды. Далее полученные данные анализируются, и LLM формирует итоговый ответ

### Какую задачу решает агент

Агент решает микро-задачу быстрого и удобного получения погодной информации. Он позволяет:
- узнать текущую температуру,
- посмотреть влажность и скорость ветра,
- получить прогноз на день,
- узнать вероятность дождя,
- получить краткую практическую интерпретацию погоды

### Архитектура агента

Агент состоит из следующих шагов:

1. Получение пользовательского запроса
2. Определение города и типа запроса
3. Получение координат города
4. Получение погодных данных из открытого API
5. Анализ и структурирование полученных данных
6. Генерация финального ответа через LLM

Схема workflow:

START -> parse_query -> geocode_location -> fetch_weather -> summarize_weather -> END

In [ ]:
!pip install langgraph

In [ ]:
import os
import requests
from typing import TypedDict, Optional, Literal
from dotenv import load_dotenv

from langgraph.graph import StateGraph, END

### Выбор API

Для получения погодных данных используется Open-Meteo API. Это открытый API, который позволяет получать:
- текущую температуру,
- влажность,
- скорость ветра,
- вероятность осадков,
- дневной прогноз.

Также Open-Meteo предоставляет geocoding API, что удобно для определения координат по названию города.

In [ ]:
# определим полный State класс (общее состояние)
class WeatherAgentState(TypedDict):
    user_query: str
    location: Optional[str]
    intent: Optional[str]
    latitude: Optional[float]
    longitude: Optional[float]
    weather_data: Optional[dict]
    structured_weather: Optional[dict]
    final_answer: Optional[str]
    error: Optional[str]

### Поддерживаемые типы запросов

Агент поддерживает 4 базовых типа запросов:

- `current_weather` — текущая погода
- `today_forecast` — прогноз на сегодня
- `rain_check` — будет ли дождь
- `detailed_weather` — температура, влажность, ветер и другие параметры

Сначала по запросу пользователя определим тип вопроса, который он задает: текущая погода или прогноз. Решим эту задачу с помощью LLM.

In [ ]:
# узел анализа пользовательского запроса
def parse_query(state: WeatherAgentState) -> WeatherAgentState:
    query = state["user_query"]

    prompt = f"""
          Ты анализируешь запрос пользователя о погоде.

          Нужно определить:
          1. location — город
          2. intent — один из:
            - current_weather
            - today_forecast
            - rain_check
            - detailed_weather

          Верни ответ строго в формате:
            location: ...
            intent: ...

          Запрос пользователя: {query}
"""

    response = llm.invoke(prompt).content.strip()

    location = None
    intent = None

    for line in response.split("\n"):
        if line.lower().startswith("location:"):
            location = line.split(":", 1)[1].strip()
        elif line.lower().startswith("intent:"):
            intent = line.split(":", 1)[1].strip()

    state["location"] = location
    state["intent"] = intent

    if not location or not intent:
        state["error"] = "Не удалось определить город или тип погодного запроса."

    return state

In [ ]:
# узел определения координат города
def geocode_location(state: WeatherAgentState) -> WeatherAgentState:
    if state.get("error"):
        return state

    location = state["location"]

    url = "https://geocoding-api.open-meteo.com/v1/search"
    params = {
        "name": location,
        "count": 1,
        "language": "ru",
        "format": "json"
    }

    response = requests.get(url, params=params, timeout=15)
    data = response.json()

    results = data.get("results")
    if not results:
        state["error"] = f"Не удалось найти координаты для города: {location}"
        return state

    state["latitude"] = results[0]["latitude"]
    state["longitude"] = results[0]["longitude"]
    return state

In [ ]:
# узел получения погодных данных. На этом этапе агент обращается к открытому API прогноза погоды и получает данные по найденным координатам
def fetch_weather(state: WeatherAgentState) -> WeatherAgentState:
    if state.get("error"):
        return state

    lat = state["latitude"]
    lon = state["longitude"]

    url = "https://api.open-meteo.com/v1/forecast"
    params = {
        "latitude": lat,
        "longitude": lon,
        "current": [
            "temperature_2m",
            "relative_humidity_2m",
            "wind_speed_10m",
            "precipitation"
        ],
        "hourly": [
            "temperature_2m",
            "relative_humidity_2m",
            "wind_speed_10m",
            "precipitation",
            "precipitation_probability"
        ],
        "daily": [
            "temperature_2m_max",
            "temperature_2m_min",
            "precipitation_sum",
            "precipitation_probability_max"
        ],
        "timezone": "auto",
        "forecast_days": 1
    }

    response = requests.get(url, params=params, timeout=15)
    data = response.json()

    state["weather_data"] = data
    return state

In [ ]:
# узел структурирования погодных данных
def analyze_weather(state: WeatherAgentState) -> WeatherAgentState:
    if state.get("error"):
        return state

    data = state["weather_data"]
    intent = state["intent"]

    current = data.get("current", {})
    daily = data.get("daily", {})
    hourly = data.get("hourly", {})

    structured = {
        "intent": intent,
        "current_temperature": current.get("temperature_2m"),
        "current_humidity": current.get("relative_humidity_2m"),
        "current_wind_speed": current.get("wind_speed_10m"),
        "current_precipitation": current.get("precipitation"),
        "temp_max": daily.get("temperature_2m_max", [None])[0] if daily.get("temperature_2m_max") else None,
        "temp_min": daily.get("temperature_2m_min", [None])[0] if daily.get("temperature_2m_min") else None,
        "precipitation_sum": daily.get("precipitation_sum", [None])[0] if daily.get("precipitation_sum") else None,
        "precipitation_probability_max": daily.get("precipitation_probability_max", [None])[0] if daily.get("precipitation_probability_max") else None,
    }

    # простая логика для evening rain check
    if intent == "rain_check":
        times = hourly.get("time", [])
        probs = hourly.get("precipitation_probability", [])

        evening_probs = []
        for t, p in zip(times, probs):
            hour = int(t[11:13])
            if 18 <= hour <= 23:
                evening_probs.append(p)

        structured["evening_rain_probability"] = max(evening_probs) if evening_probs else None

    state["structured_weather"] = structured
    return state

Напишем шаг, который будет по координатам возвращать требуемый ответ

In [ ]:
def summarize_weather(state: WeatherAgentState) -> WeatherAgentState:
    if state.get("error"):
        state["final_answer"] = state["error"]
        return state

    location = state["location"]
    user_query = state["user_query"]
    weather = state["structured_weather"]

    prompt = f"""
        Ты погодный ассистент.

        Пользователь спросил: {user_query}
        Город: {location}
        Структурированные погодные данные: {weather}

        Сформулируй краткий и понятный ответ на русском языке.
        Правила:
        - не выводи JSON;
        - отвечай естественно;
        - если уместно, добавь практический совет;
        - если вероятность дождя высокая, прямо скажи, что лучше взять зонт;
        - если ветер сильный, укажи это;
        - не придумывай данные, которых нет.
"""

    state["final_answer"] = llm.invoke(prompt).content.strip()
    return state

In [ ]:
def route_after_parse(state: WeatherAgentState) -> str:
    if state.get("error"):
        return "end"
    return "geocode"

def route_after_geocode(state: WeatherAgentState) -> str:
    if state.get("error"):
        return "end"
    return "fetch"

In [ ]:
# собираем граф
graph = StateGraph(WeatherAgentState)

graph.add_node("parse_query", parse_query)
graph.add_node("geocode_location", geocode_location)
graph.add_node("fetch_weather", fetch_weather)
graph.add_node("analyze_weather", analyze_weather)
graph.add_node("summarize_weather", summarize_weather)

graph.set_entry_point("parse_query")

graph.add_conditional_edges(
    "parse_query",
    route_after_parse,
    {
        "geocode": "geocode_location",
        "end": END
    }
)

graph.add_conditional_edges(
    "geocode_location",
    route_after_geocode,
    {
        "fetch": "fetch_weather",
        "end": END
    }
)

graph.add_edge("fetch_weather", "analyze_weather")
graph.add_edge("analyze_weather", "summarize_weather")
graph.add_edge("summarize_weather", END)

app = graph.compile()

А теперь проверяем как работает

In [ ]:
result = app.invoke({
    "user_query": "Какая сейчас погода в Берлине?",
    "location": None,
    "intent": None,
    "latitude": None,
    "longitude": None,
    "weather_data": None,
    "structured_weather": None,
    "final_answer": None,
    "error": None
})

print(result["final_answer"])

В Берлине сейчас температура 13.7°C, влажность 34%, ветер со скоростью 10.5 м/с. Дождя не ожидается, но ветер достаточно сильный, так что стоит учесть это при выходе на улицу. Если планируете быть на улице долго, одевайтесь по теме!


In [ ]:
queries = [
    "Какая сейчас температура, влажность и ветер в Москве?",
    "Будет ли дождь вечером в Казани?",
    "Дай прогноз на сегодня для Санкт-Петербурга"
]

for q in queries:
    result = app.invoke({
        "user_query": q,
        "location": None,
        "intent": None,
        "latitude": None,
        "longitude": None,
        "weather_data": None,
        "structured_weather": None,
        "final_answer": None,
        "error": None
    })
    print("Запрос:", q)
    print("Ответ:", result["final_answer"])
    print("-" * 80)

Запрос: Какая сейчас температура, влажность и ветер в Москве?
Ответ: В Москве сейчас температура воздуха 6.6°C, влажность составляет 44%, а ветер дует со скоростью 3.4 м/с. Дождя нет и не предвидится, так что зонт брать не нужно. Хороший день для прогулки на свежем воздухе!
--------------------------------------------------------------------------------
Запрос: Будет ли дождь вечером в Казани?
Ответ: Вечером в Казани дождя не ожидается, вероятность составляет 0%. Температура будет около 3.1°C, ветер легкий (4.6 м/с). Можно смело выйти на улицу без зонта!
--------------------------------------------------------------------------------
Запрос: Дай прогноз на сегодня для Санкт-Петербурга
Ответ: Сегодня в Санкт-Петербурге температура воздуха составляет около 10.1°C, с максимальным значением 11.5°C и минимальным 3.6°C. Влажность на уровне 56%, но дождя не ожидается. Однако ветер довольно сильный — скорость 13.3 м/с. Рекомендуется одеться комфортно и, возможно, потеплее, чтобы почувствовать 

In [ ]:
# и еще поиграемся
queries = [
    "Какая сейчас температура, влажность и ветер в Сеуле?",
    "Будет ли снег завтра утром в Париже?",
    "Дай прогноз на завтра для Нижнего Новгорода"
]

for q in queries:
    result = app.invoke({
        "user_query": q,
        "location": None,
        "intent": None,
        "latitude": None,
        "longitude": None,
        "weather_data": None,
        "structured_weather": None,
        "final_answer": None,
        "error": None
    })
    print("Запрос:", q)
    print("Ответ:", result["final_answer"])
    print("-" * 80)

Запрос: Какая сейчас температура, влажность и ветер в Сеуле?
Ответ: Сейчас в Сеуле температура составляет 4.7°C, влажность – 56%, а скорость ветра – 1.6 м/с. Дождя не ожидается. В такую погоду лучше одеться потеплее.
--------------------------------------------------------------------------------
Запрос: Будет ли снег завтра утром в Париже?
Ответ: Завтра утром в Париже снега не ожидается. Температура будет 15.7°C, и вероятность осадков составляет 0%. Ветер будет довольно умеренным со скоростью 13.1 м/с. Вы можете планировать свои мероприятия на улице без особых беспокойств!
--------------------------------------------------------------------------------
Запрос: Дай прогноз на завтра для Нижнего Новгорода
Ответ: Завтра в Нижнем Новгороде ожидается температура от -1.6° до 10.1°. На улице сухо, дождя не предсказывают, и вероятность осадков составляет 0%. Ветер будет умеренный со скоростью 4.2 м/с. Учитывая такие условия, можно смело планировать прогулки на улице. Просто не забудьте одетьс